# Model Comparison

Three stages, all from paired per-seed differences: every configuration saw the same three seeds and the same test set, so each seed forms a matched pair.

No numeric threshold is applied. A configuration is reported as a candidate winner only when its difference carries the **same sign on all three seeds**; the mean and standard deviation of the differences are reported alongside for magnitude. With n=3, a threshold rule would have no theoretical basis and would read as a significance test without being one -- and formal tests are not used for the same reason.

Pure analysis over the saved long-format results: no GPU, no checkpoints.

In [ ]:
!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'

In [ ]:
import pandas as pd
from comparison import stl_vs_mtl, flat_vs_mtl, strategy_pairs, efficiency_summary

long_df = pd.read_csv(f'{RESULTS_DIR}/test_results_long.csv')
long_df.model.unique(), long_df.seed.unique()

## Stage 1 -- single-task vs multi-task

Does folding the two tasks into one backbone cost either of them? Model A against each MTL variant on species, Model B against each on freshness: 6 comparison points. `direction = b_better` means the MTL variant won.

In [ ]:
stage1 = stl_vs_mtl(long_df, metric='f1_macro')
stage1.to_csv(f'{RESULTS_DIR}/comparison_stage1_stl_vs_mtl.csv', index=False)
stage1

## Stage 2 -- flat 24-class vs multi-task

Compares the problem formulation itself: predicting the pair as one 24-way label against factorising it into two heads.

In [ ]:
stage2 = flat_vs_mtl(long_df, metric='f1_macro')
stage2.to_csv(f'{RESULTS_DIR}/comparison_stage2_flat_vs_mtl.csv', index=False)
stage2

## Stage 3 -- weighting strategies against each other

In [ ]:
stage3 = pd.concat([
    strategy_pairs(long_df, 'species', 'f1_macro'),
    strategy_pairs(long_df, 'freshness', 'f1_macro'),
    strategy_pairs(long_df, 'freshness', 'qwk'),
    strategy_pairs(long_df, 'joint', 'joint_accuracy'),
], ignore_index=True)
stage3.to_csv(f'{RESULTS_DIR}/comparison_stage3_strategies.csv', index=False)
stage3

Rows with `consistent_sign = False` mean the two configurations traded places across seeds: report them as indistinguishable at this sample size rather than picking the higher mean.

In [ ]:
summary = pd.concat([stage1, stage2, stage3], ignore_index=True)
summary[summary.consistent_sign][['task', 'metric', 'model_a', 'model_b', 'direction',
                                  'mean_difference', 'std_difference']].round(4)

## Efficiency

The realistic alternative to one multi-task model is running both single-task models in sequence, so parameters and latency are compared against Model A + Model B combined.

In [ ]:
efficiency = efficiency_summary(long_df)
efficiency.to_csv(f'{RESULTS_DIR}/comparison_efficiency.csv')
efficiency